## 📦 1. Load and Prepare Dataset

In [ ]:
import pandas as pd

# Load dataset
DATA_PATH = "/home/krish/Downloads/Myntra Fasion Clothing.csv"
df = pd.read_csv(DATA_PATH)

# Drop unnecessary columns
df.drop(columns=["DiscountPrice (in Rs)", "Ratings", "Reviews", "DiscountOffer"], inplace=True)

# Rename columns for consistency
df.rename(columns={
    'URL': 'url',
    'Product_id': 'id',
    'BrandName': 'brand',
    'Category': 'category',
    'Individual_category': 'sub_category',
    'category_by_Gender': 'gender',
    'Description': 'description',
    'OriginalPrice (in Rs)': 'price',
    'SizeOption': 'size'
}, inplace=True)

# Sample products across gender and categories
def get_products(gender, sub_category, n=20):
    filtered = df[(df['sub_category'] == sub_category) & (df['gender'] == gender)].dropna()
    return filtered.sample(n=min(n, len(filtered)), random_state=42)

product_needed = ['shirts', 'jeans', 'tshirts', 'track-pants']
products = pd.concat(
    [get_products(g, p) for p in product_needed for g in ['Men', 'Women']],
    ignore_index=True
)

# Save intermediate result
products.to_csv("cleaned_product_sample.csv", index=False)


## 🖼️ 2. Scrape Product Image URLs Using Selenium

In [ ]:
from selenium import webdriver
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.common.by import By
from multiprocessing import Pool
import re, time

def extract_image_url(product_url):
    options = webdriver.FirefoxOptions()
    options.add_argument('--headless')
    driver = webdriver.Firefox(service=Service(), options=options)

    try:
        driver.get(product_url)
        time.sleep(2)
        div = driver.find_element(By.CLASS_NAME, "image-grid-image")
        style_attr = div.get_attribute("style")
        match = re.search(r'url\("([^"]+)"\)', style_attr)
        return match.group(1) if match else None
    except Exception as e:
        print(f"Error fetching {product_url}: {e}")
        return None
    finally:
        driver.quit()

def parallel_fetch(df, url_column='url', processes=4):
    urls = df[url_column].tolist()
    with Pool(processes=processes) as pool:
        image_urls = pool.map(extract_image_url, urls)
    df = df.copy()
    df['image_url'] = image_urls
    return df

products = pd.read_csv("cleaned_product_sample.csv")
final_products = parallel_fetch(products, url_column='url', processes=4)
final_products = final_products.dropna(subset=['image_url'])
final_products.to_csv("final_products.csv", index=False)


## 🧠 3. Generate Image Embeddings using Fashion-CLIP

In [ ]:
import requests
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from io import BytesIO
import torch
import numpy as np
from tqdm import tqdm

final_products = pd.read_csv("final_products.csv")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CLIPModel.from_pretrained("patrickjohncyh/fashion-clip", cache_dir='hf_models').to(device)
processor = CLIPProcessor.from_pretrained("patrickjohncyh/fashion-clip", cache_dir='hf_models')
model.eval()

def load_image_from_url(url):
    try:
        response = requests.get(url, timeout=5)
        return Image.open(BytesIO(response.content)).convert("RGB")
    except Exception as e:
        print(f"Failed to load image from {url}: {e}")
        return None

batch_size = 16
embeddings_list = []

for i in tqdm(range(0, len(final_products), batch_size)):
    batch = final_products.iloc[i:i+batch_size]
    images = [load_image_from_url(url) for url in batch['image_url']]
    valid_indices = [j for j, img in enumerate(images) if img is not None]
    valid_images = [img for img in images if img is not None]

    if valid_images:
        with torch.no_grad():
            inputs = processor(images=valid_images, return_tensors="pt", padding=True).to(device)
            image_features = model.get_image_features(**inputs)
            image_features = torch.nn.functional.normalize(image_features, p=2, dim=1).cpu().numpy()
    else:
        image_features = []

    idx = 0
    for j in range(len(batch)):
        if j in valid_indices:
            emb = image_features[idx].tolist()
            idx += 1
        else:
            emb = [0.0] * model.config.projection_dim
        embeddings_list.append(emb)

final_products['image_embedding'] = embeddings_list


## 📝 4. Generate Text Embeddings using Sentence Transformers

In [ ]:
from sentence_transformers import SentenceTransformer

cols = ['brand', 'category', 'sub_category', 'gender', 'description', 'size']

def make_product_text(row):
    return " ".join([f"{col}: {str(row[col])}" for col in cols])

final_products['text_input'] = final_products.apply(make_product_text, axis=1)

text_model = SentenceTransformer(
    'sentence-transformers/multi-qa-MiniLM-L6-cos-v1', 
    cache_folder='hf_models', device=device
)

all_embeddings = text_model.encode(
    final_products['text_input'].tolist(),
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
)

final_products['text_embedding'] = [emb.tolist() for emb in all_embeddings]
final_products.drop(columns=['text_input'], inplace=True)
final_products.to_csv("final_products.csv", index=False)


## 🔍 5. Text-to-Image Search using CLIP

In [ ]:
df = pd.read_csv("final_products.csv")
df['image_embedding'] = df['image_embedding'].apply(eval)

model = CLIPModel.from_pretrained("patrickjohncyh/fashion-clip", cache_dir="hf_models").to(device)
processor = CLIPProcessor.from_pretrained("patrickjohncyh/fashion-clip", cache_dir="hf_models")

query = "A yellow shirt for men"

with torch.no_grad():
    inputs = processor(text=query, return_tensors="pt").to(device)
    text_features = model.get_text_features(**inputs)
    text_features = torch.nn.functional.normalize(text_features, p=2, dim=1).cpu().numpy()[0]

image_embeddings = np.array(df['image_embedding'].tolist())
df['similarity'] = np.dot(image_embeddings, text_features)

top_matches = df.sort_values(by='similarity', ascending=False).head(10)
print(top_matches[['image_url', 'brand', 'category', 'similarity']])


## 🔎 6. Text-to-Text Search using Sentence Transformers

In [ ]:
from sentence_transformers import util
import torch
import numpy as np

df = pd.read_csv("final_products.csv")
df['text_embedding'] = df['text_embedding'].apply(eval)

text_model = SentenceTransformer(
    'sentence-transformers/multi-qa-MiniLM-L6-cos-v1', 
    cache_folder='hf_models', device=device
)

query = "comfortable red dress for summer"
query_emb = text_model.encode(query, convert_to_tensor=True, normalize_embeddings=True).to(device)

product_embs = torch.tensor(df['text_embedding'].tolist(), dtype=torch.float32).to(device)
scores = util.dot_score(query_emb, product_embs)[0].cpu().tolist()

df['similarity'] = scores
top_matches = df.sort_values(by='similarity', ascending=False).head(10)

for _, row in top_matches.iterrows():
    print(f"{row['similarity']:.4f} | {row['url']}")
